In [1]:
#!pip install azure-functions-durable==1.2.9

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import logging
import sys
sys.path.append('../')
logger = logging.getLogger()
logger.setLevel(logging.INFO)

In [4]:
from dotenv import load_dotenv
load_dotenv()

True

In [5]:
import azure.functions as func
import azure.durable_functions as df
import functools
from typing import Callable
from centraal_dataframework.constants import TASK_PARAM_NAME

#from centraal_dataframework.blueprints import runner
#from centraal_dataframework.resources import GreatExpectationsToolKit, get_context, get_datalake

In [6]:
STR_FMT = 'TAREA: %(name)s--%(asctime)s-%(levelname)s-%(message)s'

In [7]:
bp = df.Blueprint()

## Internal working




In [8]:
myApp = df.DFApp(http_auth_level=func.AuthLevel.ANONYMOUS)

In [9]:
import json
from dataclasses import dataclass, field
from typing import Callable, List
import azure.functions as func
import inspect
from azure.durable_functions import Blueprint, DurableOrchestrationContext


@dataclass
class TaskInfo:
    succeeded: bool
    args: dict = field(default_factory=dict)

    def to_json(self):
        return json.dumps(self.__dict__)

    @staticmethod
    def from_json(json_str):
        data = json.loads(json_str)
        return TaskInfo(**data)




class Task:
    def __init__(self, func: Callable):
        self.func = func
        self.name = func.__name__
    
    def _check_func(self):
        signature = inspect.signature(self.func)
        parameters = signature.parameters
        
        if TASK_PARAM_NAME not in parameters:
            raise TypeError(f"El parametro debe ser llamado {TASK_PARAM_NAME}")
        
        param = parameters[TASK_PARAM_NAME]
        param_type = param.annotation
        
        if not param_type==TaskInfo:
            raise ValueError(f"El parametro debe ser tipo `TaskInfo`")     
    
    def __call__(self, *args, **kwargs):

        self._check_func()

        return self

In [10]:
def check_if_func_name_is_register(bp: Blueprint, name: str):
    pass

In [11]:
class Pipeline:

    def __init__(self, name: str, tasks: List[Task], bp: Blueprint) -> None:

        self.tasks: List[Task] = tasks
        self.name = name
        self.route = f"start_{self.name}"
        self.build(bp)

    def _register_tasks(self, bp: Blueprint):

        for t in self.tasks:
            bp.activity_trigger(input_name="task_info", activity=t.name)(t.func)

    def __create_orchestator_fun(self) -> Callable:
        def orchestator(context: DurableOrchestrationContext):
            results = {}

            try:
                for t in self.tasks:
                    if not results:
                        result = yield context.call_activity(t.name, TaskInfo(True, {"some": "parameter"}))

                    else:
                        result[t.name] = yield context.call_activity(t.name, result)

                    results[t.name] = result

            except Exception as e:
                yield context.call_activity(
                    "alert",
                    TaskInfo(
                        succeeded=False,
                        args={
                            "name_task_failed": t.name,
                            "task_executed": [n for n in results.keys()],
                            "error": str(e),
                        },
                    ),
                )

                raise ValueError("Error en pipeline")

            return results

        return orchestator

    def __create_start_fun(self) -> Callable:
        async def start_orchestrator_func(req: func.HttpRequest, client):
            instance_id = await client.start_new(self.route)
            logging.info(f"Started orchestration with ID = '{instance_id}'.")
            return client.create_check_status_response(req, instance_id)

        return start_orchestrator_func

    def _create_register_orchestartor(self, bp: Blueprint):
        fun_orch = self.__create_orchestator_fun()
        bp.orchestration_trigger(context_name="context", orchestration=self.name)(fun_orch)

    def _create_register_start(self, bp: Blueprint):
        fun_start = self.__create_start_fun()
        bp.route(route=self.route)(bp.durable_client_input(client_name="client")(fun_start))

    def register_pipeline(self, bp: Blueprint):
        self._register_tasks(bp)
        self._create_register_orchestartor(bp)
        self._create_register_start(bp)

    def build(self, bp: Blueprint):
        self.register_pipeline(bp)

In [12]:
bp = df.Blueprint()
@Task
def first_step_of_pipeline(task_info: TaskInfo):
    print("first step")

pipeline = Pipeline(name = "test_pipeline", tasks=[first_step_of_pipeline, first_step_of_pipeline], bp= bp)

In [17]:
for f in bp._function_builders:
    f  = f._function
    print(f._name)

first_step_of_pipeline
first_step_of_pipeline
orchestator
start_orchestrator_func


In [14]:
app = func.FunctionApp(http_auth_level=func.AuthLevel.FUNCTION)
app.register_functions(bp)

In [16]:
fs = app.get_functions()

# when the function already exsits, I register a new activity trigger, the idea is not register but call it with the same on the ochestation
for f in fs:
    print(f)

{"scriptFile": "function_app.py", "bindings": [{"direction": "IN", "type": "activityTrigger", "name": "task_info", "activity": "first_step_of_pipeline"}]}
{"scriptFile": "function_app.py", "bindings": [{"direction": "IN", "type": "activityTrigger", "name": "task_info", "activity": "first_step_of_pipeline"}]}
{"scriptFile": "function_app.py", "bindings": [{"direction": "IN", "type": "orchestrationTrigger", "name": "context", "orchestration": "test_pipeline"}]}
{"scriptFile": "function_app.py", "bindings": [{"direction": "IN", "type": "durableClient", "name": "client"}, {"direction": "IN", "type": "httpTrigger", "name": "req", "authLevel": "FUNCTION", "route": "start_test_pipeline"}, {"direction": "OUT", "type": "http", "name": "$return"}]}


2. Then a pipeline will be an orchestator function, that calls the activity functions, We want create a way to define a pipeline very easy and internally add those tasks into the orchestation function.

This orchestation function will run everything and if any task fails, we will send an email, in this way we can group task into a `pipeline` in order to inform more that one `task` in the same email.


> note: this activity to send an email needs to be executed by another `task`, this task has to be general for all the `pipeline`. This is recomendation to comply with the recomendation: An orchestator should not impoklement logic to wait external systems of I/O tasks.



In [ ]:
@bp.route(route="startOrchestrator")
@bp.durable_client_input(client_name="client")
async def start_orchestrator(req: func.HttpRequest, client):
    instance_id = await client.start_new("my_orchestrator")
    
    logging.info(f"Started orchestration with ID = '{instance_id}'.")
    return client.create_check_status_response(req, instance_id)


3. We want to able to schedule these pipeline, so we wil create an instance of `runner` that will contains all those pipelines

> note: Is a table a proper a better way to storage this information?

The framework will have 3 `standard` functions:
1. a function that will have a timer trigger (60 minutes), that will check with the runner which pipelines needs to be executed

> note: we assume that this function will start each pipeline and with the control of [Configuration throttles](https://learn.microsoft.com/en-us/azure/azure-functions/durable/durable-functions-perf-and-scale#configuration-of-throttles) we will able to limit the concurrency, by default the framework enable one `pipeline` and one `task` each time. The important question is: Can we start all the pipelines needed and this configuration will manage the concurrency without any intervention?


also it seems nthat we need to take into account the `partitionCount`

2. a function that have an http trigger with a request containing the name of the pipeline to start

> note: the previuos question here also important, Can we start at demand a new `pipeline` and the durable fucntion framework will ad this `pipeline` to the end of the queue withoput nay intervention?

3. A function to get the status of a pipeline -> this can be useful to leverage other external monitoring acitivities. The idea is to have a http trigger.

> can we also use this to get the status of the pipeline running?